# ncountr — Nanostring nCounter Analysis in Python

**Zero-install demo** — run this notebook on [Google Colab](https://colab.research.google.com/) or [Binder](https://mybinder.org/).

This notebook walks through the full ncountr pipeline:
1. Install ncountr
2. Parse RCC files
3. QC checks
4. Normalization
5. Differential expression
6. Gene set scoring
7. Export to AnnData (scverse ecosystem)

In [ ]:
%pip install ncountr[crossplatform] -q

## 1. Install ncountr

## 2. Parse RCC files

ncountr ships with synthetic example RCC files. On Colab, we clone the repo to access them.

In [ ]:
import os
from pathlib import Path

# On Colab, clone the repo to get example data
if not Path("sample_rcc").exists():
    if Path("/content").exists():  # Colab
        !git clone --depth 1 https://github.com/princello/ncountr.git /tmp/ncountr
        rcc_dir = "/tmp/ncountr/examples/sample_rcc"
    else:
        rcc_dir = "sample_rcc"  # local / Binder
else:
    rcc_dir = "sample_rcc"

print(f"RCC directory: {rcc_dir}")
print(f"Files: {sorted(os.listdir(rcc_dir))}")

In [ ]:
import ncountr

# Parse RCC files — sample IDs are extracted from the filename
exp = ncountr.read_rcc(
    rcc_dir,
    sample_id_pattern=r"(\d+)",
    sample_meta={
        "1": {"group": "treated"},
        "2": {"group": "treated"},
        "3": {"group": "control"},
    },
)
print(exp)
print(f"Genes: {exp.genes[:5]}...")
print(f"Samples: {exp.samples}")

## 3. Quality Control

QC checks FOV ratio, positive control linearity (R²), housekeeping stability, and negative control background.

In [ ]:
qc_results = ncountr.qc(exp)
qc_results

In [ ]:
from ncountr.plotting.qc_plots import plot_qc
fig = plot_qc(exp)
fig.tight_layout()

## 4. Normalization

Positive control + housekeeping normalization (`pos_hk`) is the recommended method. This corrects for both technical variation (positive controls) and input RNA amount (housekeeping genes).

In [ ]:
ncountr.normalize(exp, method="pos_hk")

print("Raw counts (first 5 genes):")
print(exp.raw_counts.head())
print("\nNormalized counts (first 5 genes):")
print(exp.normalized.head())

## 5. Differential Expression

Compare treated vs control using Mann-Whitney U test with FDR correction.

In [ ]:
de_results = ncountr.de(
    exp,
    group_a=["1", "2"],  # treated
    group_b=["3"],        # control
    test="mannwhitneyu",
)
de_results.sort_values("pvalue")

In [ ]:
from ncountr.plotting.de_plots import plot_volcano
fig = plot_volcano(
    de_results,
    highlight_genes=ncountr.get_gene_set("IFN_JAKSTAT"),
    highlight_label="IFN/JAK-STAT",
    title="Treated vs Control",
)

## 6. Gene Set Scoring

Score each sample for IFN/JAK-STAT pathway activity using built-in gene sets.

In [ ]:
scores = ncountr.score_gene_set(exp, gene_set="IFN_JAKSTAT")
print("IFN/JAK-STAT pathway scores per sample:")
print(scores)

## 7. Export to AnnData

Convert to AnnData for integration with the scverse ecosystem (scanpy, squidpy, etc.).

In [ ]:
adata = ncountr.to_anndata(exp)
print(adata)
print(f"\nobs columns: {list(adata.obs.columns)}")
print(f"var columns: {list(adata.var.columns)}")
print(f"layers: {list(adata.layers.keys())}")
print(f"\nNormalized X[0, :5]: {adata.X[0, :5]}")
print(f"Raw layer[0, :5]:    {adata.layers['raw'][0, :5]}")

## Next steps

- **Download real data from GEO**: `ncountr fetch-geo GSE275334`
- **Run a full pipeline**: `ncountr run config.yaml`
- **Read the docs**: [ncountr.readthedocs.io](https://ncountr.readthedocs.io)
- **Report issues**: [github.com/princello/ncountr/issues](https://github.com/princello/ncountr/issues)